# 🧱 Part 01: Tokenizer Basics: How Text Becomes Numbers

> **Goal for this part**: Understand what happens before an LLM can "read" text. We'll first run a real tokenizer to see what it outputs, then rebuild character-level, word-level, and subword-level tokenizers by hand.

You can think of a tokenizer as the "translator" that sits in front of the model.

An LLM does not read raw strings. Your input text is first split into tokens, then mapped into token IDs. Let's start by running a real tokenizer and looking at what the model input actually looks like.


In [1]:
# First, run a real tokenizer and inspect the outputs
try:
    import tiktoken

    modern_tokenizer = tiktoken.get_encoding("gpt2")
    print("Real tokenizer: GPT-2 byte-level BPE")
    print(f"Vocabulary size: {modern_tokenizer.n_vocab}")
except Exception as e:
    modern_tokenizer = None
    print("Could not load tiktoken. Install it to run the real-tokenizer demo.")
    print(f"Reason: {e}")


def inspect_modern_tokenizer(text):
    """Print tokens, token IDs, and underlying bytes from a real tokenizer."""
    ids = modern_tokenizer.encode(text)
    tokens = [modern_tokenizer.decode([tok_id]) for tok_id in ids]

    print("=" * 68)
    print(f"Original text: {text!r}")
    print(f"Token IDs: {ids}")
    print(f"Number of tokens: {len(ids)}")
    print("Inspect each token:")
    for i, (tok_id, token) in enumerate(zip(ids, tokens)):
        visible = token.replace("\n", "\\n").replace("\t", "\\t")
        token_bytes = list(modern_tokenizer.decode_single_token_bytes(tok_id))
        print(f"  {i:02d} | id={tok_id:<6} | token={visible!r} | bytes={token_bytes}")


if modern_tokenizer is not None:
    examples = [
        "the cat sat on the mat",
        "A tokenizer turns text into numbers.",
    ]

    for example in examples:
        inspect_modern_tokenizer(example)

    print("=" * 68)
    print("Key observation: the model consumes token IDs, not raw strings.")
    print("Next, we'll start from the simplest character-level tokenizer and implement it ourselves.")


Real tokenizer: GPT-2 byte-level BPE
Vocabulary size: 50257
Original text: 'the cat sat on the mat'
Token IDs: [1169, 3797, 3332, 319, 262, 2603]
Number of tokens: 6
Inspect each token:
  00 | id=1169   | token='the' | bytes=[116, 104, 101]
  01 | id=3797   | token=' cat' | bytes=[32, 99, 97, 116]
  02 | id=3332   | token=' sat' | bytes=[32, 115, 97, 116]
  03 | id=319    | token=' on' | bytes=[32, 111, 110]
  04 | id=262    | token=' the' | bytes=[32, 116, 104, 101]
  05 | id=2603   | token=' mat' | bytes=[32, 109, 97, 116]
Original text: 'A tokenizer turns text into numbers.'
Token IDs: [32, 11241, 7509, 4962, 2420, 656, 3146, 13]
Number of tokens: 8
Inspect each token:
  00 | id=32     | token='A' | bytes=[65]
  01 | id=11241  | token=' token' | bytes=[32, 116, 111, 107, 101, 110]
  02 | id=7509   | token='izer' | bytes=[105, 122, 101, 114]
  03 | id=4962   | token=' turns' | bytes=[32, 116, 117, 114, 110, 115]
  04 | id=2420   | token=' text' | bytes=[32, 116, 101, 120, 116]
  05 |

## Learning map

In this part you only need to hold on to 4 ideas:

1. **A Tokenizer is a translator**: it turns text into token IDs, and can turn IDs back into text.
2. **Character-level is robust**: it almost never has "unknown words", but the sequence becomes very long.
3. **Word-level is short**: far fewer tokens, but the vocabulary can explode and you will hit OOV.
4. **Subword-level is a compromise**: common pieces stay together, rare words get split. This is the direction of BPE in the next part.


## 1. First, be precise: what are Token and Tokenizer?

Let's start with a clear definition.

**Token**: the smallest text unit the model processes in one step.

A token can be a character, a word, or a common subword piece.

```text
Character-level token:  "cat" -> ["c", "a", "t"]
Word-level token:       "the cat" -> ["the", "cat"]
Subword-level token:    "playing" -> ["play", "ing"]
```

**Tokenizer**: a tool that translates back and forth between "text" and a "token-ID sequence".

It does two jobs:

```text
text   --encode--> token IDs
IDs    --decode--> text
```

Why do we need a tokenizer at all?

Because the neural network inside an LLM only consumes numeric tensors, not Python strings. Your input like `"the cat"` must be split into tokens first, then each token must be mapped to an ID.

For example, a character-level split might look like:

```text
"the cat" -> ["t", "h", "e", " ", "c", "a", "t"]
          -> [5, 12, 3, 0, 6, 1, 8]
```

Important: these numbers are just labels. `12` is not "bigger" or "better" than `3`. It is simply the ID assigned to a different token.

So the real question in this part is: **how should we split text into tokens?**

We'll start from a tiny corpus and try three choices: character-level, word-level, and subword-level.

Along the way, let's also meet something very common in modern LLMs: **special tokens**.

Normal tokens come from the text itself, like `the`, `cat`, `ing`.
Special tokens are hand-defined control symbols, for example:

```text
<BOS>  begin of sequence
<EOS>  end of sequence
<PAD>  padding positions
```

They also have their own IDs. The difference is that normal tokens represent content, while special tokens represent "boundaries" and "rules".
Later, when we build Mini-GPT, you will also see special tokens like `<think>` to mark a reasoning region.


## 2. Prepare a tiny corpus

A tokenizer needs to know what kinds of text it will see, so we'll prepare a small corpus.

We'll start with English because spaces make word boundaries easy to see. Once you understand the idea, Chinese is not a different world, just trickier splitting rules.


In [2]:
# Our tiny corpus - we intentionally choose simple sentences with repeated patterns
# This makes it easier to see what gets merged later in BPE
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

print(f"Corpus size: {len(corpus)} sentences")
for i, s in enumerate(corpus):
    print(f"  [{i}] {s}")

total_chars = sum(len(s) for s in corpus)
print(f"\nTotal characters: {total_chars}")
print(f"Total words (split by spaces): {sum(len(s.split()) for s in corpus)}")


Corpus size: 10 sentences
  [0] the cat sat on the mat
  [1] the dog sat on the log
  [2] the cat and the dog
  [3] i love my cat
  [4] i love my dog
  [5] the cat is cute
  [6] the dog is happy
  [7] the mat is soft
  [8] the log is hard
  [9] cats and dogs are friends

Total characters: 175
Total words (split by spaces): 46


**First: what a computer sees as "text"**

On the screen you see `the cat`, but at the lowest level the computer sees character codes.

```
Text:    t    h    e         c    a    t
        |    |    |    |    |    |    |
ASCII:  116  104  101  32   99   97   116
```

So the real tokenizer question is not "can we turn text into numbers". The question is: **how fine should we split?**

- Split by characters: one letter per token
- Split by words: one word per token
- Split by subwords: common pieces become tokens, like `ing`, `the`, `cat`

Let's try them one by one.


## 3. Character-level

### Split by characters

The simplest idea is: **one character = one token**.

```
"the cat"
  -> split into characters
['t', 'h', 'e', ' ', 'c', 'a', 't']
  -> lookup IDs
[5, 12, 3, 0, 6, 1, 8]
```

The vocabulary (vocab) is just a dictionary from "character -> ID".

The advantages are obvious: simple, stable, and it almost never runs into unknown words. Let's implement one by hand.


In [3]:
class CharTokenizer:
    """
    Character-level tokenizer: one character = one token.

    Only two core methods: encode and decode.
    """

    def __init__(self):
        # stoi = string-to-id: map a character to an integer ID
        # itos = id-to-string: map an integer ID back to a character
        self.stoi = {}
        self.itos = {}

    def train(self, texts):
        """Build a vocab by collecting all characters that appear in the corpus.

        Steps:
        1. Concatenate all texts into one long string
        2. Deduplicate with a set to get all unique characters
        3. Sort them and assign a unique ID to each character
        """
        all_text = "".join(texts)
        chars = sorted(list(set(all_text)))

        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}

        print("=== Character-level tokenizer trained ===")
        print(f"Vocab size: {len(self.stoi)} tokens")
        print(f"Vocab items: {chars}")

    def encode(self, text):
        """Encode: text -> list of token IDs."""
        return [self.stoi[ch] for ch in text]

    def decode(self, ids):
        """Decode: list of token IDs -> text."""
        return "".join([self.itos[i] for i in ids])


In [4]:
# Train and test
char_tokenizer = CharTokenizer()
char_tokenizer.train(corpus)

# Try one sentence
text = "the cat"
ids = char_tokenizer.encode(text)
recovered = char_tokenizer.decode(ids)

print(f"\n=== Encode / decode test ===")
print(f"Original text: '{text}'")
print(f"Token IDs: {ids}")
print(f"Decoded back: '{recovered}'")
print(f"\nKey observation: {len(text)} characters -> {len(ids)} tokens (the same count!)")
print("Each character becomes one token. There is no compression.")


=== Character-level tokenizer trained ===
Vocab size: 20 tokens
Vocab items: [' ', 'a', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'y']

=== Encode / decode test ===
Original text: 'the cat'
Token IDs: [16, 7, 4, 0, 2, 1, 16]
Decoded back: 'the cat'

Key observation: 7 characters -> 7 tokens (the same count!)
Each character becomes one token. There is no compression.


### The problems with character-level

Character-level tokenization is easy to understand, but it comes with two issues.

**Problem 1: sequences get too long.**

`"the cat"` has 7 characters, which means 7 tokens. A 1000-character article becomes a 1000-token sequence.

Self-Attention inside LLMs is very sensitive to sequence length. If length grows 10x, compute can approach 100x.

**Problem 2: meaning gets shattered.**

`cat` is a single concept, but character-level splits it into `c`, `a`, `t`. The model must learn from scratch that these three characters often form "cat".

So a natural thought is: what if we split by words directly?


## 4. Word-level

### Split by words

The word-level idea is: **one word = one token**.

```
"the cat sat on the mat"
  -> split by spaces
['the', 'cat', 'sat', 'on', 'the', 'mat']
  -> lookup IDs
[0, 1, 2, 3, 0, 4]
```

The advantage is that sequences become much shorter: `the cat` is only 2 tokens.

But word-level has a fatal problem: **what if we see a word we have never seen before?**

If the vocab does not contain `cats`, the tokenizer cannot assign it an ID. This is the OOV (out-of-vocabulary) problem.


In [5]:
class WordTokenizer:
    """
    Word-level tokenizer: split by spaces, one word = one token.

    The structure is the same as CharTokenizer; only the splitting granularity changes from characters to words.
    """

    def __init__(self):
        self.stoi = {}
        self.itos = {}

    def train(self, texts):
        """Collect all words that appear in the corpus."""
        all_words = set()
        for text in texts:
            words = text.split()
            all_words.update(words)

        all_words = sorted(list(all_words))
        self.stoi = {w: i for i, w in enumerate(all_words)}
        self.itos = {i: w for i, w in enumerate(all_words)}

        print("=== Word-level tokenizer trained ===")
        print(f"Vocab size: {len(self.stoi)} words")
        print(f"Vocab items: {all_words}")

    def encode(self, text):
        """Split by spaces and look up each word."""
        return [self.stoi[w] for w in text.split()]

    def decode(self, ids):
        """Map IDs back to words and join with spaces."""
        return " ".join([self.itos[i] for i in ids])


In [6]:
# Train and test
word_tokenizer = WordTokenizer()
word_tokenizer.train(corpus)

text = "the cat sat on the mat"
ids = word_tokenizer.encode(text)
recovered = word_tokenizer.decode(ids)

print(f"\n=== Encode / decode test ===")
print(f"Original text: '{text}'")
print(f"Token IDs: {ids}")
print(f"Decoded back: '{recovered}'")
print(f"\nKey observation: {len(text.split())} words -> {len(ids)} tokens (much shorter than character-level)")


=== Word-level tokenizer trained ===
Vocab size: 20 words
Vocab items: ['and', 'are', 'cat', 'cats', 'cute', 'dog', 'dogs', 'friends', 'happy', 'hard', 'i', 'is', 'log', 'love', 'mat', 'my', 'on', 'sat', 'soft', 'the']

=== Encode / decode test ===
Original text: 'the cat sat on the mat'
Token IDs: [19, 2, 17, 16, 19, 14]
Decoded back: 'the cat sat on the mat'

Key observation: 6 words -> 6 tokens (much shorter than character-level)


In [7]:
# But word-level tokenization has a fatal problem: what about OOV (out-of-vocabulary) words?
print("=== OOV (Out Of Vocabulary) demo ===")
print(f"Current vocab: {list(word_tokenizer.stoi.keys())}")
print()

try:
    test_text = "the elephant sat"
    print(f"Try to encode: '{test_text}'")
    ids = word_tokenizer.encode(test_text)
    print(f"Result: {ids}")
except KeyError as e:
    # Let's see what happens when it fails
    words = test_text.split()
    for w in words:
        if w in word_tokenizer.stoi:
            print(f"  OK '{w}' -> ID {word_tokenizer.stoi[w]} (in vocab)")
        else:
            print(f"  X  '{w}' -> not in vocab! KeyError!")
    print(f"\nConclusion: when we hit an unseen word like 'elephant', the whole program crashes.")
    print("In the real world, you can never pre-enumerate every possible word in the vocabulary.")


=== OOV (Out Of Vocabulary) demo ===
Current vocab: ['and', 'are', 'cat', 'cats', 'cute', 'dog', 'dogs', 'friends', 'happy', 'hard', 'i', 'is', 'log', 'love', 'mat', 'my', 'on', 'sat', 'soft', 'the']

Try to encode: 'the elephant sat'
  OK 'the' -> ID 19 (in vocab)
  X  'elephant' -> not in vocab! KeyError!
  OK 'sat' -> ID 17 (in vocab)

Conclusion: when we hit an unseen word like 'elephant', the whole program crashes.
In the real world, you can never pre-enumerate every possible word in the vocabulary.


## 5. Subword-level

Let's put the three choices side by side:

| Scheme | Vocab size | Sequence length | OOV risk | Intuition |
|:---|:---|:---|:---|:---|
| Character-level | tiny | long | almost none | robust, but too fine-grained |
| Word-level | huge | short | common | clear, but too rigid |
| Subword-level | controllable | medium | rare | compromise |

What subword tokenization wants is:

```
"the cat sat"      -> [the, cat, sat]
"cats and dogs"    -> [cat, s, and, dog, s]
"unbelievable"     -> [un, believ, able]
```

Common words stay intact, rare words get split into smaller pieces. This keeps the vocabulary from exploding, and new words do not break the system.


### BPE intuition

The core move of BPE is simple: **find the most frequent adjacent character pair, and merge it into a new token**.

```
Start: one character per token
['t', 'h', 'e', ' ', 'c', 'a', 't', ...]
        -> count adjacent pairs
('t', 'h') appears many times
        -> merge
new token: 'th'
        -> repeat count + merge
```

If you repeat this many times, the vocabulary will gradually grow from characters into subwords like `th`, `the`, `ing`.

In the next part we will implement BPE from scratch and print every merge step.


---

## Summary

Make sure you understand these:

1. Tokenizer is a translator between text and token IDs
2. `encode` maps text -> IDs, `decode` maps IDs -> text
3. The vocab is the set of all valid tokens, each with a unique ID
4. Character-level is simple and robust, but sequences are long and meaning is fragmented
5. Word-level sequences are short, but vocab is huge and OOV is common
6. Subword-level is a compromise: merge frequent pieces, split rare words

Next: implement BPE by hand and watch how the vocab "grows".


---

## Exercises: Tokenizer Basics

These 3 exercises fall into two categories:

1. **Core idea**: the essence of a tokenizer is `text <-> token IDs`.
2. **Modern LLM practice**: real models often use special tokens, attention masks, and fixed-length batches.

> **About using AI**: it is fine to ask for hints, step-by-step reasoning, or direction checks, but avoid asking an AI to directly complete the exercise for you. The point is to make the mapping from text to IDs with your own hands.


### Exercise 1: core mechanism - what does `encode` do?

Looking tokens up in a vocab and turning them into IDs is the most essential tokenizer operation.

**Hint**: `stoi[token]` maps a token to its ID.


In [ ]:
# Exercise 1: fill in encode
stoi = {"the": 0, "cat": 1, "sat": 2}
tokens = ["the", "cat", "sat"]

# TODO: replace the triple-quoted placeholder with your code
ids = """Replace this with code that maps each token in `tokens` to its ID."""

assert not isinstance(ids, str), "Please replace the placeholder inside the triple quotes."
assert ids == [0, 1, 2], ids
print("OK Exercise 1 passed: you remember that encode is simply token -> ID")


### Exercise 2: modern practice - add special tokens

Modern LLMs do not only consume normal text tokens. They often use `<BOS>` to mark the beginning and `<EOS>` to mark the end.

**Hint**: many model inputs look like `[BOS] + content_tokens + [EOS]`.


In [ ]:
# Exercise 2: fill in special tokens
stoi = {"<BOS>": 0, "<EOS>": 1, "the": 2, "cat": 3}
tokens = ["the", "cat"]

# TODO: replace the triple-quoted placeholder with your code
input_ids = """Replace this with code that adds <BOS> and <EOS> IDs around the content."""

assert not isinstance(input_ids, str), "Please replace the placeholder inside the triple quotes."
assert input_ids == [0, 2, 3, 1], input_ids
print("OK Exercise 2 passed: you know modern LLM inputs often include special tokens")


### Exercise 3: modern practice - padding and attention masks

In a batch, sequences have different lengths. We usually pad them to the same length, and use an attention mask to mark which positions are real tokens.

**Hint**: real tokens have mask `1`, padding positions have mask `0`.


In [ ]:
# Exercise 3: fill in padding + attention mask
PAD_ID = 0
ids = [5, 6, 7]
max_len = 5

# TODO: replace the triple-quoted placeholder with your code
padded_ids = """Replace this with code that pads `ids` to `max_len`."""
attention_mask = """Replace this with code that marks real tokens vs PAD positions."""

assert not isinstance(padded_ids, str), "Please replace the placeholder for padded_ids."
assert not isinstance(attention_mask, str), "Please replace the placeholder for attention_mask."
assert padded_ids == [5, 6, 7, 0, 0], padded_ids
assert attention_mask == [1, 1, 1, 0, 0], attention_mask
print("OK Exercise 3 passed: you understand why attention_mask is a common tokenizer output")
